In [1]:
from mlpui.model_loader import load_torch_file

In [2]:
sd,meta = load_torch_file("../models/mlp/uma-s-1p1.pt",return_metadata=True)

In [3]:
def unet_prefix_from_state_dict(state_dict):
    candidates = ["model.diffusion_model.",
                  "model.model.",
                  "net.",
                  "module.backbone.",
                  ]
    counts = {k: 0 for k in candidates}
    for k in state_dict:
        for c in candidates:
            if k.startswith(c):
                counts[c] += 1
                break

    top = max(counts, key=counts.get)
    if counts[top] > 5:
        return top
    else:
        return "model." #aura flow and others


In [4]:
prefix = unet_prefix_from_state_dict(sd)
prefix

'module.backbone.'

In [5]:
def calculate_parameters(sd, prefix=""):
    params = 0
    for k in sd.keys():
        if k.startswith(prefix):
            w = sd[k]
            params += w.nelement()
    return params

In [6]:
calculate_parameters(sd,prefix=prefix)

146533699

In [7]:
def weight_dtype(sd, prefix=""):
    dtypes = {}
    for k in sd.keys():
        if k.startswith(prefix):
            w = sd[k]
            dtypes[w.dtype] = dtypes.get(w.dtype, 0) + w.numel()

    if len(dtypes) == 0:
        return None

    return max(dtypes, key=dtypes.get)
weight_dtype(sd,prefix=prefix)

torch.float32

In [8]:
from mlpui.model_detection import model_config_from_unet,count_blocks
mlp_config = model_config_from_unet(sd,prefix)
print(mlp_config)

{'model_type': 'uma', 'num_blocks': 4, 'has_mole': True, 'datasets': ['oc20', 'omol', 'omat', 'odac', 'omc']}
